# Lab 10-2: Structured Streaming with IoT Data

In this lab we are going to take a quick look at how to use Structured Streaming API to build a streaming application based on a simulated IOT (Internet of Things) dataset. We want to compute real-time metrics like running counts and windowed counts on a stream of timestamped IoT events (e.g. the Opening and close events) that often comes in Json format.

**Datasets Used:**
- `/databricks-datasets/structured-streaming/events/`  
  Sample IoT event data in JSON format, each record contains `time` and `action` fields.

**Tables/Views Created:**
- `events`  
  Temporary view of the static DataFrame loaded from the JSON files.
- `lab10_2.counts`  
  Database/table used for SQL queries and visualization.

**Database/Schema/Volume Created:**
- Catalog: `msbabigdata`
- Schema: `msbabigdata.spark`
- Volume: `msbabigdata.spark.lab10_2`

## Step 1: Inspect Data
We have some sample action data as files in `/databricks-datasets/structured-streaming/events/` which we are going to use to build this appication. 

It is generally a good practice to test your streaming code first on static data because you cannot stop or pause a streaming processor as easily. 

Let's take a look at the contents of this directory.

### 1. Create the volume needed for this lab.

- Catalog (`msbabigdata`), Schema (`spark`), Volume (`lab10_2`)

In [ ]:
%sql
create catalog if not exists msbabigdata;
create schema if not exists msbabigdata.spark;
create volume if not exists msbabigdata.spark.lab10_2;


### 2. Create the database and table

- Create a database lab10_2 (if  it does not already exists)
- Drop (if exists)  and recreate a table `counts` in the database, with the following fields

```
  action STRING,
  window STRUCT<start: TIMESTAMP, end: TIMESTAMP>,
  count BIGINT
```


In [ ]:
%sql

CREATE DATABASE IF NOT EXISTS lab10_2; 
DROP TABLE IF EXISTS lab10_2.counts;
CREATE TABLE IF NOT EXISTS lab10_2.counts(
    action STRING, 
    window STRUCT<start: TIMESTAMP, end: TIMESTAMP>,
    count BIGINT
    );

### 3. Inspect the data files

Our data set is a shared dataset from Databricks at: `/databricks-datasets/structured-streaming/events/`

- view the files in the data directory (using `dbutils.fs.ls()` or Linux command `ls -l`)
- preview the file format (using dbutils.fs.head or Linux command `head`)

In [ ]:
!ls -l /databricks-datasets/structured-streaming/events/

total 3575
-rwxrwxrwx 1 nobody nogroup 72530 Aug  6  2020 file-0.json
-rwxrwxrwx 1 nobody nogroup 73025 Aug  6  2020 file-10.json
-rwxrwxrwx 1 nobody nogroup 72999 Aug  6  2020 file-11.json
-rwxrwxrwx 1 nobody nogroup 72987 Aug  6  2020 file-12.json
-rwxrwxrwx 1 nobody nogroup 73006 Aug  6  2020 file-13.json
-rwxrwxrwx 1 nobody nogroup 73003 Aug  6  2020 file-14.json
-rwxrwxrwx 1 nobody nogroup 73007 Aug  6  2020 file-15.json
-rwxrwxrwx 1 nobody nogroup 72978 Aug  6  2020 file-16.json
-rwxrwxrwx 1 nobody nogroup 73008 Aug  6  2020 file-17.json
-rwxrwxrwx 1 nobody nogroup 73002 Aug  6  2020 file-18.json
-rwxrwxrwx 1 nobody nogroup 73014 Aug  6  2020 file-19.json
-rwxrwxrwx 1 nobody nogroup 72961 Aug  6  2020 file-1.json
-rwxrwxrwx 1 nobody nogroup 72987 Aug  6  2020 file-20.json
-rwxrwxrwx 1 nobody nogroup 72983 Aug  6  2020 file-21.json
-rwxrwxrwx 1 nobody nogroup 73009 Aug  6  2020 file-22.json
-rwxrwxrwx 1 nobody nogroup 72985 Aug  6  2020 file-23.json
-rwxrwxrwx 1 nobody nogroup 730

In [ ]:
!head /databricks-datasets/structured-streaming/events/file-0.json

{"time":1469501107,"action":"Open"}
{"time":1469501147,"action":"Open"}
{"time":1469501202,"action":"Open"}
{"time":1469501219,"action":"Open"}
{"time":1469501225,"action":"Open"}
{"time":1469501234,"action":"Open"}
{"time":1469501245,"action":"Open"}
{"time":1469501246,"action":"Open"}
{"time":1469501248,"action":"Open"}
{"time":1469501256,"action":"Open"}


Each line in the file contains JSON record with two fields - `time` and `action`. Let's try to analyze these files interactively first

## Step 2: Develop/test the business logic using SQL/DataFrame APIs first



### 4. Load and aggregate the static data

In this step, we develop and test the business logic using DataFrame APIs. 

- Load the json dataset into a dataframe `df` with provided schema `time timestamp, action string`
    - This is because, with streaming, you should not ask spark to read sample data and infer schema. Not only inferSchema is unreliabe but also time-consuming. 
- Create a temp view `events` so that we may use SQL API to aggregate event data.
- Count the number of events by `action` and `window`, where window is a sliding window with a duration of 1 hour, and a sliding duration of 5 minutes.
- Save the aggregate data into a table `counts` in the `lab10_2` database, using the `overwrite` mode.

> Note that the document for the [window function](https://spark.apache.org/docs/latest/api/python/_modules/pyspark/sql/functions.html#window) shows that `
window(timeColumn, windowDuration, slideDuration=None)` 
> where duration arguments can be specified in the form of "5 seconds", "1 week", "1 hours", and so on.


In [ ]:
schema = "time timestamp, action string"
data_path = "/databricks-datasets/structured-streaming/events/"
path = "/Volumes/msbabigdata/spark/lab10_2"
df = (spark.read
     .schema(schema)
     .json(data_path)
     .createOrReplaceTempView("events") 
)

countDf = spark.sql("""
select action, 
       window,
       count(*) as count
from ( 
      select action, window(time, "1 hour","30 minutes") AS window
      from events
)
group by action, window
""")

countDf.write.mode("overwrite").saveAsTable("lab10_2.counts")

### 5. Use SQL to obtain counts of "open" and "close" requests for each window. 

- select `action`, `time`, `count`, where is the hour part of the window's starting time. 
- format `time` using `date_format(date, "MMM-dd HH")`
- order results by `time`, `action`

- Visualize the counts result using a bar chart
  - `action` will serve as series.


In [ ]:
%sql
SELECT action, date_format(window.start, "MMM-dd HH:mm") as time, count
from lab10_2.counts
order by time, action;

action | time | count
Close | Jul-26 02:00 | 11
Open | Jul-26 02:00 | 179
Close | Jul-26 02:30 | 116
Open | Jul-26 02:30 | 680
Close | Jul-26 03:00 | 344
Open | Jul-26 03:00 | 1001
Close | Jul-26 03:30 | 581
Open | Jul-26 03:30 | 993
Close | Jul-26 04:00 | 815
Open | Jul-26 04:00 | 999
Close | Jul-26 04:30 | 987
Open | Jul-26 04:30 | 1008
Close | Jul-26 05:00 | 1003
Open | Jul-26 05:00 | 1000
Close | Jul-26 05:30 | 968
Open | Jul-26 05:30 | 993
Close | Jul-26 06:00 | 1011
Open | Jul-26 06:00 | 993
Close | Jul-26 06:30 | 1039
Open | Jul-26 06:30 | 998
... (208 rows total)

## Step 3: Stream Processing 
Now that we have analyzed the data interactively, let's convert this to a streaming app.


### 6. Create a streaming application with the following activities:

The business logic is the same as the one we have developed and tested interactively, but the source and destination specification are different.

- The Structured Streaming DataFrame is configured to obtain 1 file per trigger
- Output to the same table, using the complete `mode`.
- use the `/Volumes/msbabigdata/spark/lab10_2/checkpoint` directory as the checkpoint location.
- trigger should be set to `availableNow` : this is the only mode supported in serverless. 
- Be sure to delete and recreate the checkpoint directory before hand.




In [ ]:
!rm -rf /Volumes/msbabigdata/spark/lab10_2/checkpoint
!mkdir /Volumes/msbabigdata/spark/lab10_2/checkpoint

In [ ]:
schema = "time timestamp, action string"
data_path = "/databricks-datasets/structured-streaming/events/"
path = "/Volumes/msbabigdata/spark/lab10_2"
df = (spark.readStream
      .option("maxFilesPerTrigger", 1)
     .schema(schema)
     .json(data_path)
     .createOrReplaceTempView("events") 
)

countDf = spark.sql("""
select action, 
       window,
       count(*) as count
from ( 
      select action, window(time, "1 hour","30 minutes") AS window
      from events
)
group by action, window
""")

(countDf
.writeStream
.trigger(availableNow=True)
.option("checkpointlocation", path+"/checkpoint")
.outputMode("complete")
.toTable("lab10_2.counts")
)

### 7. Query the table

- run the same query as in the static model
- view results in bar charts
- repeat the same query every few seconds to see how it changes.

In [ ]:
%sql
SELECT action, date_format(window.start, "MMM-dd HH:mm") as time, count
from lab10_2.counts
order by time, action;

action | time | count
Close | Jul-26 02:00 | 11
Open | Jul-26 02:00 | 179
Close | Jul-26 02:30 | 116
Open | Jul-26 02:30 | 680
Close | Jul-26 03:00 | 344
Open | Jul-26 03:00 | 1001
Close | Jul-26 03:30 | 581
Open | Jul-26 03:30 | 993
Close | Jul-26 04:00 | 815
Open | Jul-26 04:00 | 999
Close | Jul-26 04:30 | 987
Open | Jul-26 04:30 | 1008
Close | Jul-26 05:00 | 1003
Open | Jul-26 05:00 | 1000
Close | Jul-26 05:30 | 968
Open | Jul-26 05:30 | 993
Close | Jul-26 06:00 | 1011
Open | Jul-26 06:00 | 993
Close | Jul-26 06:30 | 1039
Open | Jul-26 06:30 | 998
... (208 rows total)

### Clean Up

- Interrupt the Streaming Processing if it is still running.
- Drop the volumes, tables/databases.

In [ ]:
%sql

DROP VOLUME IF EXISTS lab10_2;
DROP DATABASE IF EXISTS lab10_2 CASCADE;